# Garmin FIT Payload Exploration

Purpose:

- Import raw Garmin running activity files in FIT format.
- Parse FIT messages into three analytical entities:
  - `sessions`: one row per run, used for run-level metrics.
  - `records`: second-by-second telemetry, used for pace, heart rate, cadence, elevation, and split-style analysis.
  - `events`: activity event messages, used to identify timer boundaries and recovery heart rate.
- Inspect available fields across sessions, records, and events.
- Validate which fields are reliable enough for bronze ingestion.
- Save representative exploration outputs for schema design.

This notebook is exploratory. Reusable Garmin authentication, download, parsing, and file IO logic should live in `ingest/garmin/**`; the notebook should only orchestrate discovery and inspect outputs.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

In [ ]:
from ingest.garmin.paths import get_garmin_fit_dir
from ingest.garmin.parser import parse_fit_files

fit_dir = get_garmin_fit_dir()
fit_paths = sorted(fit_dir.glob("*.fit"))

print(f"FIT directory: {fit_dir}")
print(f"FIT files found: {len(fit_paths)}")

if len(fit_paths) == 0:
    print("No FIT files found. Exiting.")
    print("Run:")
    print("    uv run python scripts/download_garmin_fit.py --destination local --mode range-overwrite --start-date <date> --end-date <date>")
else:
    print("Parsing FIT files...")
    parsed = parse_fit_files(fit_paths)
    sessions = parsed["sessions"]
    events = parsed["events"]
    records = parsed["records"]
    print(f"Parsed {len(records)} records from {len(fit_paths)} FIT files.")

In [ ]:
entity_shapes = pd.DataFrame(
    [
        {
            "entity": "sessions",
            "rows": len(sessions),
            "columns": len(sessions.columns),
            "unique_runs": sessions["run_id"].nunique() if "run_id" in sessions.columns else None,
        },
        {
            "entity": "events",
            "rows": len(events),
            "columns": len(events.columns),
            "unique_runs": events["run_id"].nunique() if "run_id" in events.columns else None,
        },
        {
            "entity": "records",
            "rows": len(records),
            "columns": len(records.columns),
            "unique_runs": records["run_id"].nunique() if "run_id" in records.columns else None,
        },
    ]
)

entity_shapes

### Validate field reliability

In [ ]:
def column_inventory(df: pd.DataFrame, entity: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "entity": entity,
            "column": df.columns,
            "dtype": [str(dtype) for dtype in df.dtypes],
            "non_null_count": df.notna().sum().values,
            "null_count": df.isna().sum().values,
            "null_pct": [df[column].isna().mean() * 100 for column in df.columns],
            "unique_count": [df[column].nunique(dropna=True) for column in df.columns],
        }
    ).sort_values(["null_pct", "column"])

field_inventory = pd.concat(
    [
        column_inventory(sessions, "sessions"),
        column_inventory(events, "events"),
        column_inventory(records, "records"),
    ],
    ignore_index=True,
)

# field_inventory

In [ ]:
candidate_fields = {
    "sessions": [
        "run_id",
        "timestamp",
        "start_time",
        "total_elapsed_time",
        "total_timer_time",
        "total_distance",
        "sport",
        "sub_sport",
        "avg_heart_rate",
        "max_heart_rate",
        "enhanced_avg_speed",
        "avg_speed",
        "enhanced_max_speed",
        "max_speed",
        "total_ascent",
        "total_descent",
        "total_calories",
        "total_training_effect",
        "total_anaerobic_training_effect",
        "avg_cadence",
        "max_cadence",
        "avg_running_cadence",
        "max_running_cadence",
    ],
    "events": [
        "run_id",
        "timestamp",
        "event",
        "event_type",
        "data",
    ],
    "records": [
        "run_id",
        "timestamp",
        "distance",
        "heart_rate",
        "enhanced_speed",
        "enhanced_altitude",
        "cadence",
        "temperature",
        "position_lat",
        "position_long",
        "position_lat_deg",
        "position_long_deg",
        "stance_time",
        "vertical_oscillation",
        "vertical_ratio",
        "step_length",
    ],
}

In [ ]:
candidate_inventory = pd.concat(
    [
        field_inventory[
            field_inventory["entity"].eq(entity)
            & field_inventory["column"].isin(fields)
        ]
        for entity, fields in candidate_fields.items()
    ],
    ignore_index=True,
)

In [ ]:
FIELD_CLASSIFICATION_RULES = {
    "required_max_null_pct": 0.0,
    "recommended_max_null_pct": 20.0,
    "optional_max_null_pct": 80.0,
}

def classify_field(null_pct: float) -> str:
    if null_pct == 0:
        return "required_or_stable"
    if null_pct <= FIELD_CLASSIFICATION_RULES["recommended_max_null_pct"]:
        return "recommended"
    if null_pct <= FIELD_CLASSIFICATION_RULES["optional_max_null_pct"]:
        return "optional"
    return "exclude_now"

candidate_inventory = candidate_inventory.copy()
candidate_inventory["reliability_class"] = candidate_inventory["null_pct"].apply(classify_field)

candidate_inventory = candidate_inventory.sort_values(["entity", "reliability_class", "null_pct", "column"]).reset_index(drop=True)
candidate_inventory

### Validate sessions

In [ ]:
session_quality_checks = {
    "session_rows": len(sessions),
    "unique_runs": sessions["run_id"].nunique(),
    "duplicate_run_ids": int(sessions["run_id"].duplicated().sum()),
}

session_quality_checks

In [ ]:
session_preview_columns = [
    column
    for column in [
        "run_id",
        "start_time",
        "timestamp",
        "sport",
        "sub_sport",
        "total_distance",
        "total_timer_time",
        "total_elapsed_time",
        "avg_heart_rate",
        "max_heart_rate",
        "enhanced_avg_speed",
        "avg_speed",
        "total_ascent",
        "total_descent",
    ]
    if column in sessions.columns
]

sessions[session_preview_columns].head()

### Validate records

In [ ]:
records_per_run = (
    records.groupby("run_id")
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

records_per_run.describe()

In [ ]:
record_quality_checks = {
    "record_rows": len(records),
    "unique_runs": records["run_id"].nunique(),
    "timestamp_null_pct": f"{records['timestamp'].isna().mean() * 100:.4f}%" if "timestamp" in records.columns else None,
    "distance_null_pct": f"{records['distance'].isna().mean() * 100:.4f}%" if "distance" in records.columns else None,
    "heart_rate_null_pct": f"{records['heart_rate'].isna().mean() * 100:.4f}%" if "heart_rate" in records.columns else None,
    "enhanced_speed_null_pct": f"{records['enhanced_speed'].isna().mean() * 100:.4f}%" if "enhanced_speed" in records.columns else None,
}

record_quality_checks

### Validate events and recovery HR

In [ ]:
event_counts = (
    events.groupby(["event", "event_type"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

event_counts

In [ ]:
recovery_events = events[
    events["event"].astype(str).str.contains("recovery_hr", case=False, na=False)
].copy().reset_index(drop=True)

if "data" in recovery_events.columns:
    recovery_events["recovery_hr"] = pd.to_numeric(
        recovery_events["data"],
        errors="coerce",
    )

recovery_events.head()

In [ ]:
recovery_event_count_distribution = (
    recovery_events.groupby("run_id")
    .size()
    .value_counts()
    .rename_axis("recovery_event_count")
    .reset_index(name="run_count")
    .sort_values("recovery_event_count")
    .reset_index(drop=True)
)

recovery_event_count_distribution

In [ ]:
recovery_coverage = {
    "runs_with_recovery_events": recovery_events["run_id"].nunique(),
    "recovery_coverage_pct": f"{(recovery_events['run_id'].nunique() / sessions['run_id'].nunique() * 100):.2f}%"
    if sessions["run_id"].nunique() > 0
    else None,
}

recovery_coverage

In [ ]:
last_record_per_run = (
    records.sort_values(["run_id", "timestamp"])
    .groupby("run_id", as_index=False)
    .tail(1)
)

last_record_columns = [
    column
    for column in ["run_id", "timestamp", "heart_rate", "distance"]
    if column in last_record_per_run.columns
]

last_records = last_record_per_run[last_record_columns].rename(
    columns={
        "timestamp": "last_record_timestamp",
        "heart_rate": "final_record_heart_rate",
        "distance": "final_record_distance",
    }
)

last_records.head()

In [ ]:
if recovery_events.empty:
    print("No recovery events found. Cannot validate recovery HR drop.")
elif "recovery_hr" not in recovery_events.columns:
    print("recovery_events exists, but `recovery_hr` has not been derived yet.")
    display(recovery_events.head(20))
else:
    recovery_validation = recovery_events.merge(last_records, on="run_id", how="left")

    if "final_record_heart_rate" in recovery_validation.columns:
        recovery_validation["final_hr_minus_recovery_hr"] = (
            recovery_validation["final_record_heart_rate"]
            - recovery_validation["recovery_hr"]
        )

    display(
        recovery_validation[
            [
                column
                for column in [
                    "run_id",
                    "last_record_timestamp",
                    "recovery_hr",
                    "final_record_heart_rate",
                    "final_hr_minus_recovery_hr",
                    "final_record_distance",
                ]
                if column in recovery_validation.columns
            ]
        ].head()
    )

### Save representative exploration outputs

In [ ]:
from ingest.garmin.paths import get_garmin_exploration_dir

exploration_dir = get_garmin_exploration_dir()
print(exploration_dir)

#### field inventory

In [ ]:
field_inventory_path = exploration_dir / "fit_field_inventory.csv"
candidate_inventory_path = exploration_dir / "fit_candidate_field_inventory.csv"

field_inventory.to_csv(field_inventory_path, index=False)
candidate_inventory.to_csv(candidate_inventory_path, index=False)

print(f"Field inventory saved to: {field_inventory_path}")
print(f"Candidate field inventory saved to: {candidate_inventory_path}")

#### representative samples

In [ ]:
sessions_sample_path = exploration_dir / "sessions.sample.csv"
events_sample_path = exploration_dir / "events.sample.csv"
records_sample_path = exploration_dir / "records.sample.csv"

sessions.head(100).to_csv(sessions_sample_path, index=False)
events.head(500).to_csv(events_sample_path, index=False)
records.head(5000).to_csv(records_sample_path, index=False)

print(f"Sessions sample dataset saved to: {sessions_sample_path}")
print(f"Events sample dataset saved to: {events_sample_path}")
print(f"Records sample dataset saved to: {records_sample_path}")

#### recovery HR sample

In [ ]:
recovery_events_path = exploration_dir / "recovery_events.sample.csv"

recovery_events.to_csv(recovery_events_path, index=False)

print(f"Recovery events dataset saved to: {recovery_events_path}")

#### schema recommendation draft

In [ ]:
schema_recommendation = candidate_inventory[
    [
        "entity",
        "column",
        "dtype",
        "non_null_count",
        "null_pct",
        "unique_count",
        "reliability_class",
    ]
].sort_values(["entity", "reliability_class", "null_pct", "column"])

schema_recommendation_path = exploration_dir / "fit_bronze_schema_recommendation.csv"

schema_recommendation.to_csv(schema_recommendation_path, index=False)

print(f"Schema recommendation saved to: {schema_recommendation_path}")

#### run coverage summary

In [ ]:
coverage_summary = pd.DataFrame(
    [
        {
            "metric": "session_rows",
            "value": len(sessions),
        },
        {
            "metric": "session_unique_runs",
            "value": sessions["run_id"].nunique(),
        },
        {
            "metric": "event_rows",
            "value": len(events),
        },
        {
            "metric": "event_unique_runs",
            "value": events["run_id"].nunique(),
        },
        {
            "metric": "record_rows",
            "value": len(records),
        },
        {
            "metric": "record_unique_runs",
            "value": records["run_id"].nunique(),
        },
        {
            "metric": "recovery_event_rows",
            "value": len(recovery_events),
        },
        {
            "metric": "recovery_event_unique_runs",
            "value": recovery_events["run_id"].nunique(),
        },
    ]
)

coverage_summary_path = exploration_dir / "fit_coverage_summary.csv"

coverage_summary.to_csv(coverage_summary_path, index=False)

print(f"Coverage summary saved to: {coverage_summary_path}")

## Exploration conclusion

This notebook confirms that Garmin FIT files can be parsed into three useful analytical entities:

- `sessions`: run-level facts suitable for consistency, volume, and summary fitness metrics.
- `records`: high-frequency telemetry suitable for pace/heart-rate analysis, splits, and within-run patterns.
- `events`: event messages suitable for timer state changes and recovery heart rate extraction.

The next engineering step is to define bronze tables for:

- `bronze_garmin_fit_sessions`
- `bronze_garmin_fit_records`
- `bronze_garmin_fit_events`

Bronze ingestion should preserve source-level values with minimal transformation. Metric derivation such as pace, HR-band efficiency, and recovery HR drop should happen in silver or gold models.